# Imports

In [5]:
from datasets import load_from_disk
import random
from gLM.dataset import Uniref90ArrowDatasetForFASTA
from gLM.tokenizers import PhyloTokenizerLoader


# Tokenizer

In [6]:
tokenizer = PhyloTokenizerLoader("./phylo_char_tokenizer_updated")

tokenizer.add_special_tokens({"bos_token": "[BOS]"})
print(tokenizer.bos_token, tokenizer.bos_token_id)

tokenizer.save_pretrained("./phylo_char_tokenizer_with_bos")


[BOS] 27


('./phylo_char_tokenizer_with_bos/tokenizer_config.json',
 './phylo_char_tokenizer_with_bos/special_tokens_map.json',
 './phylo_char_tokenizer_with_bos/tokenizer.json')

In [7]:
tokenizer = PhyloTokenizerLoader("./phylo_char_tokenizer_with_bos")

print("bos_token:", tokenizer.bos_token)
print("bos_token_id:", tokenizer.bos_token_id)
print("pad_token_id:", tokenizer.pad_token_id)
print("sep_token_id:", tokenizer.sep_token_id)
print("cls_token_id:", tokenizer.cls_token_id)
print("vocab_size:", tokenizer.vocab_size)

bos_token: [BOS]
bos_token_id: 27
pad_token_id: 0
sep_token_id: 3
cls_token_id: 2
vocab_size: 27


In [8]:
print(tokenizer.tokenize("[BOS]"))
print(tokenizer.convert_tokens_to_ids("[BOS]"))
print(tokenizer.decode([tokenizer.bos_token_id], skip_special_tokens=False))

['[BOS]']
27
[BOS]


# Dataset

In [ ]:
train_ds = load_from_disk("/gpfs/data/brandeslab/Data/uniref/uniref90_clusters_arrow/train")
print("num_rows:", train_ds.num_rows)

num_rows: 53405637


In [4]:
dataset_path = "/gpfs/data/brandeslab/Data/uniref/uniref90_clusters_arrow/train"
fasta_path = "/gpfs/data/brandeslab/Data/uniref/uniref100.fasta"
idx_db_path = "/gpfs/data/brandeslab/User/as12267/uniref100.idx"

ds = Uniref90ArrowDatasetForFASTA(
    dataset_path=dataset_path,
    training_type="phylo_encoder_decoder",
    fasta_path=fasta_path,
    idx_db_path=idx_db_path,
)

batch_raw = [ds[i] for i in range(4)]

for i, (s1, s2) in enumerate(batch_raw):
    print(f"\nExample {i}")
    print("s1:", s1[:120])
    print("s2:", s2[:120])
    print("len(s1):", len(s1), "len(s2):", len(s2))


Example 0
s1: MALGVPISVCLLFNAMTALTEEAAVIVTPPSSVQQSNWTVNKTEDDYSEGPIALRFSHPCLEDHNSYCINGMCAFHHELEKAICRCYTGYTGERCEHLTLTSYAVDSYEKYIAIGIGVGL
s2: MALGVPISVCLLFNAMTALTEEAAVIVTPPNAVQQSNWTVNKTEDDYAEGPIALRFSHPCLEDHNSYCINGMCAFHHELEKAICRCYTGYTGERCEHLTLTSYAVDSYEKYIAIGIGVGL
len(s1): 153 len(s2): 153

Example 1
s1: PSELPSHPLPQFLKAAESSDHNVLRIRFRTSSTNGLLFLAAGQASYLLLELHAGRLQLKLDLGSGEQMLESERGTQLNDLAWHSVEVHHAQLNVTLTVDKNSHTIVNMPGSHHNLNIVDG
s2: MDSAAKSTKKLLLRGLLWWSLLVQVASGASFYGDGFVQLKAAESSDHNVLRIRFRTSSTNGLLFLAAGQASYLLLELHAGRLQLKLDLGSGEQMLESERGTQLNDLAWHSVEVHHAQLNV
len(s1): 2361 len(s2): 2263

Example 2
s1: MITRTVTPSESGKKLHRYLCTLMPNYPLGQIYKMIDQGKVRVNGKRKKQNYEMASGDELTIFVDEETFQRSVGAEKKPKFVGIPANIDVVYEDDELLVVNKPAGLLTHPDRTEQKDTLIA
s2: MITRTVTPSESGKKLHRYLCSLMPNYPLGQIYKMIDQGKVRVNGKRKKQNYEMASGDELTIFVDEETFQRSAGAEKKPKFIGIPANIDVVYEDDELLVVNKPAGLLTHPDRTEQKDTLIT
len(s1): 309 len(s2): 309

Example 3
s1: MAIGRRRVRGGISARVRRGWELVTTALALLAGVYLVLTTIEPTRIVLERYVGKLDLQGLVALVAVMLEIATIAIYQHGRDVRALRALITERQRRDVTHSLADVLAVMGGTGTGRRARQVE
s

# Collator

In [9]:
import torch


class NotebookPackedCollator:
    """
    Collator for conditional next-token training of P(seq1 | seq2)
    using packed encoder-only inputs:

        seq2 [SEP] [BOS] seq1

    Attention pattern to be implemented later in the model via prefix_mask:
        - seq2 -> seq2 : bidirectional
        - seq2 -> seq1 : blocked
        - target block ([BOS] + seq1) -> seq2 : allowed
        - target block ([BOS] + seq1) -> target block : causal

    Labels:
        - -100 on seq2, [SEP], padding
        - [BOS] predicts first token of seq1
        - each seq1 token predicts the next seq1 token
        - final seq1 token gets label -100
    """

    def __init__(self, tokenizer, max_seq_len=256):
        self.tokenizer = tokenizer
        self.max_seq_len = max_seq_len

        self.pad_id = tokenizer.pad_token_id
        self.sep_id = tokenizer.sep_token_id
        self.bos_id = tokenizer.bos_token_id

        if self.pad_id is None:
            raise ValueError("Tokenizer must have pad_token_id")
        if self.sep_id is None:
            raise ValueError("Tokenizer must have sep_token_id")
        if self.bos_id is None:
            raise ValueError("Tokenizer must have bos_token_id")

    def __call__(self, batch):
        # batch: List[(s1, s2)]
        seq1s, seq2s = zip(*batch)

        # tokenize separately, no auto special tokens
        tok1 = self.tokenizer(
            list(seq1s),
            add_special_tokens=False,
            padding=False,
            truncation=True,
        )
        tok2 = self.tokenizer(
            list(seq2s),
            add_special_tokens=False,
            padding=False,
            truncation=True,
        )

        packed_ids = []
        meta = []

        for ids1, ids2 in zip(tok1["input_ids"], tok2["input_ids"]):
            # packed format: seq2 [SEP] [BOS] seq1
            prefix = ids2 + [self.sep_id] + [self.bos_id]

            remaining = self.max_seq_len - len(prefix)
            if remaining > 0:
                ids1 = ids1[:remaining]
                ids = prefix + ids1
            else:
                ids1 = []
                ids = prefix[:self.max_seq_len]

            s2_len = len(ids2[: max(0, self.max_seq_len - 2)])
            sep_pos = s2_len
            bos_pos = sep_pos + 1
            seq1_start = bos_pos + 1
            seq1_len = max(0, len(ids) - seq1_start)

            packed_ids.append(ids)
            meta.append(
                {
                    "s2_len": s2_len,
                    "sep_pos": sep_pos,
                    "bos_pos": bos_pos,
                    "seq1_start": seq1_start,
                    "seq1_len": seq1_len,
                }
            )

        B = len(batch)
        L = max(len(x) for x in packed_ids)

        input_ids = torch.full((B, L), self.pad_id, dtype=torch.long)
        padding_mask = torch.zeros((B, L), dtype=torch.long)
        labels = torch.full((B, L), -100, dtype=torch.long)
        prefix_mask = torch.zeros((B, L, L), dtype=torch.bool)

        for b, ids in enumerate(packed_ids):
            cur_len = len(ids)

            input_ids[b, :cur_len] = torch.tensor(ids, dtype=torch.long)
            padding_mask[b, :cur_len] = 1

            s2_len = meta[b]["s2_len"]
            sep_pos = meta[b]["sep_pos"]
            bos_pos = meta[b]["bos_pos"]
            seq1_start = meta[b]["seq1_start"]
            seq1_len = meta[b]["seq1_len"]

            # -------------------------
            # Labels
            # -------------------------
            # [BOS] predicts first token of seq1
            if seq1_len >= 1:
                labels[b, bos_pos] = input_ids[b, seq1_start]

            # seq1[i] predicts seq1[i+1]
            if seq1_len >= 2:
                labels[b, seq1_start : seq1_start + seq1_len - 1] = input_ids[
                    b, seq1_start + 1 : seq1_start + seq1_len
                ]

            # -------------------------
            # Prefix / block attention mask
            # -------------------------
            # valid regions:
            #   seq2:        [0, s2_len)
            #   [SEP]:       sep_pos
            #   target block [BOS]+seq1: [bos_pos, seq1_start+seq1_len)

            target_start = bos_pos
            target_end = seq1_start + seq1_len

            # seq2 -> seq2 and [SEP]
            if s2_len > 0:
                prefix_mask[b, 0:s2_len, 0 : s2_len + 1] = True

            # [SEP] -> seq2 and [SEP]
            if sep_pos < cur_len:
                prefix_mask[b, sep_pos, 0 : s2_len + 1] = True

            # target block -> seq2 and [SEP]
            if target_end > target_start:
                prefix_mask[b, target_start:target_end, 0 : s2_len + 1] = True

                # target block causal within itself
                for q in range(target_start, target_end):
                    prefix_mask[b, q, target_start : q + 1] = True

            # padding rows/cols remain all False

        return {
            "input_ids": input_ids,         # [B, L]
            "padding_mask": padding_mask,   # [B, L]
            "labels": labels,               # [B, L]
            "prefix_mask": prefix_mask,     # [B, L, L]
            "meta": meta,
        }

In [10]:
collator = NotebookPackedCollator(tokenizer, max_seq_len=256)
batch_out = collator(batch_raw)

for k, v in batch_out.items():
    if k == "meta":
        print(k, len(v))
    else:
        print(k, v.shape)

Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


input_ids torch.Size([4, 256])
padding_mask torch.Size([4, 256])
labels torch.Size([4, 256])
prefix_mask torch.Size([4, 256, 256])
meta 4


In [11]:
i = 0
print(tokenizer.decode(batch_out["input_ids"][i], skip_special_tokens=False))
print(batch_out["labels"][i])
print(batch_out["prefix_mask"][i].int())
print(batch_out["meta"][i])

M A L G V P I S V C L L F N A M T A L T E E A A V I V T P P N A V Q Q S N W T V N K T E D D Y A E G P I A L R F S H P C L E D H N S Y C I N G M C A F H H E L E K A I C R C Y T G Y T G E R C E H L T L T S Y A V D S Y E K Y I A I G I G V G L L I S G F L A I F Y C Y I R K R C L K L K S P Y N I C S G G R P L [SEP] [BOS] M A L G V P I S V C L L F N A M T A L T E E A A V I V T P P S S V Q Q S N W T V N K T E D D Y S E G P I A L R F S H P C L E D H N S Y C I N G M C A F H H E L E K A I C R C Y T G Y T G E R C E H L T L T
tensor([-100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100,
        -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100,
        -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100,
        -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100,
        -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100,
        -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100, -100,


In [12]:
i = 0
tokens = tokenizer.convert_ids_to_tokens(batch_out["input_ids"][i])
label_ids = batch_out["labels"][i].tolist()

for pos, (tok, lab) in enumerate(zip(tokens, label_ids)):
    lab_tok = "IGN" if lab == -100 else tokenizer.convert_ids_to_tokens([lab])[0]
    print(pos, tok, "->", lab_tok)

0 M -> IGN
1 A -> IGN
2 L -> IGN
3 G -> IGN
4 V -> IGN
5 P -> IGN
6 I -> IGN
7 S -> IGN
8 V -> IGN
9 C -> IGN
10 L -> IGN
11 L -> IGN
12 F -> IGN
13 N -> IGN
14 A -> IGN
15 M -> IGN
16 T -> IGN
17 A -> IGN
18 L -> IGN
19 T -> IGN
20 E -> IGN
21 E -> IGN
22 A -> IGN
23 A -> IGN
24 V -> IGN
25 I -> IGN
26 V -> IGN
27 T -> IGN
28 P -> IGN
29 P -> IGN
30 N -> IGN
31 A -> IGN
32 V -> IGN
33 Q -> IGN
34 Q -> IGN
35 S -> IGN
36 N -> IGN
37 W -> IGN
38 T -> IGN
39 V -> IGN
40 N -> IGN
41 K -> IGN
42 T -> IGN
43 E -> IGN
44 D -> IGN
45 D -> IGN
46 Y -> IGN
47 A -> IGN
48 E -> IGN
49 G -> IGN
50 P -> IGN
51 I -> IGN
52 A -> IGN
53 L -> IGN
54 R -> IGN
55 F -> IGN
56 S -> IGN
57 H -> IGN
58 P -> IGN
59 C -> IGN
60 L -> IGN
61 E -> IGN
62 D -> IGN
63 H -> IGN
64 N -> IGN
65 S -> IGN
66 Y -> IGN
67 C -> IGN
68 I -> IGN
69 N -> IGN
70 G -> IGN
71 M -> IGN
72 C -> IGN
73 A -> IGN
74 F -> IGN
75 H -> IGN
76 H -> IGN
77 E -> IGN
78 L -> IGN
79 E -> IGN
80 K -> IGN
81 A -> IGN
82 I -> IGN
83 C -> IGN
84